### Telegram Bot

In [1]:
import sys
import os

# Check Python version
print(f"Python Version: `{sys.version}`")  # Detailed version info
print(f"Base Python location: `{sys.base_prefix}`")
print(f"Current Environment location: `{os.path.basename(sys.prefix)}`", end='\n\n')

Python Version: `3.12.8 (tags/v3.12.8:2dc476b, Dec  3 2024, 19:30:04) [MSC v.1942 64 bit (AMD64)]`
Base Python location: `C:\Users\LMT\AppData\Local\Programs\Python\Python312`
Current Environment location: `.venv`



In [2]:
import os
from telegram import Bot
from telegram.constants import ParseMode
import asyncio

In [3]:
class TelegramNotifier:
    def __init__(self, bot_token, chat_id):
        self.bot_token = bot_token
        self.chat_id = chat_id
        self.bot = Bot(token=bot_token)
    
    async def send_message_async(self, message):
        """Send message asynchronously"""
        try:
            await self.bot.send_message(
                chat_id=self.chat_id,
                text=message,
                parse_mode=ParseMode.MARKDOWN,
                disable_web_page_preview=False
            )
            return True
        except Exception as e:
            print(f"Error sending message: {e}")
            return False
    
    async def send_message(self, message):
        """Synchronous wrapper for sending messages"""
        return await self.send_message_async(message)
    
    def format_flight_notification(self, flight_data, search_params):
        """Format flight data into nice notification"""
        
        price = flight_data['price_gbp']
        
        # Generate Skyscanner link
        from_code = search_params['from_code']
        to_code = search_params['to_code']
        depart_date = search_params['depart_date'].replace('-', '')
        return_date = search_params['return_date'].replace('-', '')
        
        skyscanner_link = (
            f"https://www.skyscanner.net/transport/flights/"
            f"{from_code.lower()}/{to_code.lower()}/"
            f"{depart_date[2:]}/{return_date[2:]}/"
            f"?adultsv2=1&cabinclass=economy"
        )
        
        # Build message
        message = f"*FLIGHT DEAL FOUND!*\n\n"
        message += f"*Price:* £{price:.2f}\n"
        message += f"*Route:* {from_code} → {to_code}\n"
        message += f"*Dates:* {search_params['depart_date']} to {search_params['return_date']}\n\n"
        
        # Outbound
        message += f"*OUTBOUND*\n"
        message += f"  {flight_data['out_origin']} → {flight_data['out_destination']}\n"
        message += f"  Airline: {flight_data['out_airline']}\n"
        message += f"  Duration: {flight_data['out_duration_min']} min\n"
        message += f"  Stops: {flight_data['out_stops']}\n"
        
        if flight_data['out_transit_airports']:
            message += f"  Via: {', '.join(flight_data['out_transit_airports'])}\n"
        
        message += f"\n"
        
        # Return
        message += f"*RETURN*\n"
        message += f"  {flight_data['ret_origin']} → {flight_data['ret_destination']}\n"
        message += f"  Airline: {flight_data['ret_airline']}\n"
        message += f"  Duration: {flight_data['ret_duration_min']} min\n"
        message += f"  Stops: {flight_data['ret_stops']}\n"
        
        if flight_data['ret_transit_airports']:
            message += f"  Via: {', '.join(flight_data['ret_transit_airports'])}\n"
        
        message += f"\n"
        message += f"[Book on Skyscanner]({skyscanner_link})\n"
        
        return message
    
    def send_flight_deals(self, flights_df, search_params, max_flights=3):
        """Send top flight deals as notifications"""
        if flights_df.empty:
            message = "No flights found for your search."
            self.send_message(message)
            return
        
        # Sort by price
        top_flights = flights_df.nsmallest(max_flights, 'price_gbp')
        
        # Send summary first
        summary = f"*FLIGHT DEAL ALERT*\n\n"
        summary += f"Found {len(flights_df)} flights\n"
        summary += f"Cheapest: £{flights_df['price_gbp'].min():.2f}\n"
        summary += f"Average: £{flights_df['price_gbp'].mean():.2f}\n\n"
        summary += f"Sending top {len(top_flights)} deals..."
        
        self.send_message(summary)
        
        # Send each deal
        for idx, row in top_flights.iterrows():
            flight_message = self.format_flight_notification(
                row.to_dict(), 
                search_params
            )
            self.send_message(flight_message)
            
            # Small delay between messages
            import time
            time.sleep(1)

In [4]:
import os
from dotenv import load_dotenv
# from src.telegram_notifier import TelegramNotifier

load_dotenv()

def test_simple_message():
    """Test 1: Send a simple message"""
    bot_token = os.getenv('TELEGRAM_BOT_TOKEN')
    chat_id = os.getenv('TELEGRAM_CHAT_ID')
    
    if not bot_token or not chat_id:
        print("Error: Missing TELEGRAM_BOT_TOKEN or TELEGRAM_CHAT_ID in .env")
        return
    
    notifier = TelegramNotifier(bot_token, chat_id)
    
    message = "*Test Message*\n\nYour FlightDealHunter bot is working!"
    
    print("Sending test message...")
    success = notifier.send_message(message)
    
    if success:
        print("Success! Check your Telegram app.")
    else:
        print("Failed to send message.")


def test_flight_notification():
    """Test 2: Send a formatted flight deal"""
    bot_token = os.getenv('TELEGRAM_BOT_TOKEN')
    chat_id = os.getenv('TELEGRAM_CHAT_ID')
    
    notifier = TelegramNotifier(bot_token, chat_id)
    
    # Sample flight data
    sample_flight = {
        'price_gbp': 1428.22,
        'out_origin': 'LHR',
        'out_destination': 'SGN',
        'out_airline': 'Air China',
        'out_duration_min': 1305,
        'out_stops': 1,
        'out_transit_airports': ['PEK'],
        'ret_origin': 'SGN',
        'ret_destination': 'LHR',
        'ret_airline': 'Air China',
        'ret_duration_min': 1400,
        'ret_stops': 1,
        'ret_transit_airports': ['PEK']
    }
    
    search_params = {
        'from_code': 'LON',
        'to_code': 'SGN',
        'depart_date': '2025-12-20',
        'return_date': '2026-01-10'
    }
    
    message = notifier.format_flight_notification(sample_flight, search_params)
    
    print("Sending flight notification...")
    success = notifier.send_message(message)
    
    if success:
        print("Success! Check your Telegram for the flight deal.")

In [5]:
print("="*60)
print("TESTING TELEGRAM BOT")
print("="*60)

print("\nTest 1: Simple Message")
test_simple_message()

print("\nTest 2: Flight Deal Notification")
test_flight_notification()

TESTING TELEGRAM BOT

Test 1: Simple Message
Sending test message...
Success! Check your Telegram app.

Test 2: Flight Deal Notification


C:\Users\LMT\AppData\Local\Temp\ipykernel_69196\1408040812.py:6: RuntimeWarning: coroutine 'TelegramNotifier.send_message' was never awaited
  test_simple_message()


Sending flight notification...
Success! Check your Telegram for the flight deal.


C:\Users\LMT\AppData\Local\Temp\ipykernel_69196\1408040812.py:9: RuntimeWarning: coroutine 'TelegramNotifier.send_message' was never awaited
  test_flight_notification()
